# BT-WOA-SA: reproduce the paper on Google Colab

This notebook installs the code, recomputes all statistics, tables and figures from the stored
raw results, and can rerun any experiment. Set `REPO_URL` to the address of the public repository,
or upload the repository folder to your Google Drive and set `DRIVE_DIR` instead.

In [ ]:
REPO_URL = ""          # for example "https://github.com/<user>/bt-woa-sa.git"
DRIVE_DIR = ""         # for example "/content/gdrive/MyDrive/bt-woa-sa" if you uploaded the folder

import os, sys
if REPO_URL:
    !git clone -q {REPO_URL} /content/bt-woa-sa
    ROOT = "/content/bt-woa-sa"
else:
    from google.colab import drive
    drive.mount("/content/gdrive")
    ROOT = DRIVE_DIR
%cd {ROOT}
!pip install -q -r requirements.txt
print("ready:", os.getcwd())

## 1. Statistics, tables and figures from the stored results (a few minutes)

In [ ]:
!python analyze_final.py
!python make_tables.py
!python make_figures.py
import pandas as pd
s = pd.read_csv("results/final/stats/summary.csv")
t = s[s.scenario == "E1_periodic"].pivot_table(index="label", columns="metric", values="mean")
t[["FND", "COV90", "packets_delivered", "PDR", "energy_per_packet_mJ", "malicious_CH_share_pre_HND"]].round(3)

In [ ]:
from IPython.display import Image, display
for f in ["fig_tradeoff_fnd_vs_delivered", "fig_alive_nodes", "fig_attacker_share", "fig_scalability", "fig_ns3_validation"]:
    display(Image(f"results/final/figures/{f}.png", width=800))

## 2. Rerun simulations (optional)

`run_final.py` skips runs whose results already exist. To rerun an experiment from scratch, move
its file out of `results/final/rows/` first (for example `E3.jsonl`). The stored results stay in
Drive when you work from `DRIVE_DIR`, so an interrupted run continues where it stopped.

In [ ]:
!python run_final.py --status
# !python run_final.py --only E3 --budget 3000     # uncomment to run experiment E3 for up to 50 minutes

## 3. Intel Berkeley Lab experiment (optional)

Download `data.txt.gz` and `mote_locs.txt` from http://db.csail.mit.edu/labdata/labdata.html into
a folder and set `INTEL_DIR` to it.

In [ ]:
INTEL_DIR = ""        # folder with data.txt.gz and mote_locs.txt
if INTEL_DIR:
    !INTEL_DIR="{INTEL_DIR}" python run_final.py --only E6 --budget 100000